# Phase 0 — Setup

**Goal:** get everything ready to run on a free GPU. By the end you'll have (1) the
code, (2) the libraries, and (3) the dataset saved permanently in your Google Drive.

### The three tools, one sentence each
- **GitHub** stores our *code*; Colab copies ("clones") it.
- **Google Colab** runs the code on Google's computers, *with a free GPU*.
- **Google Drive** is permanent storage; Colab forgets everything when closed, so the *dataset* lives in Drive.

Run each cell with **Shift+Enter**, top to bottom.

## Step 1 — Turn on the GPU

Colab menu: **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**,
then run the next cell to confirm.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "— none (set Runtime → T4 GPU)")

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

Here it also downloads the code and connects Drive for the first time.

In [ ]:
# === Bootstrap — RUN ME FIRST (set REPO_URL to your repo) ===
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os, sys, subprocess

def _find_repo_root():
    # Are we already inside the repo (or just above the notebooks/ folder)?
    for cand in (".", "..", "pm25-visual-aq"):
        if os.path.isdir(os.path.join(cand, "src")):
            return os.path.abspath(cand)
    return None

_root = _find_repo_root()
if _root is None:                       # fresh Colab session: clone the code
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pm25-visual-aq"], check=True)
    _root = os.path.abspath("pm25-visual-aq")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

print("repo root:", _root, "| Colab:", IN_COLAB)

## Step 2 — Load the dataset and save it to Drive (run once)

This downloads PM25Vision (~1 GB) on Google's fast network and saves it to your Drive.
Every later session detects the saved copy and skips the download.

In [ ]:
from datasets import load_dataset, load_from_disk
from src.config import load_config

cfg = load_config()
drive_path = cfg["data"]["drive_path"]

if os.path.exists(drive_path):
    print("Already saved at", drive_path)
    ds = load_from_disk(drive_path)
else:
    print("Downloading", cfg["data"]["hf_repo"], "…")
    ds = load_dataset(cfg["data"]["hf_repo"])
    ds.save_to_disk(drive_path)
    print("Saved to", drive_path)

print(ds)

## Step 3 — Confirm it worked

You should see ~11,219 rows total, AQI ranging ~1–530, and a street photo.

In [ ]:
import numpy as np, io
from PIL import Image
import matplotlib.pyplot as plt

train = ds["train"]
pm = np.array(train["pm25"])
print("columns:", train.column_names)
print("rows: train=%d test=%d" % (len(ds["train"]), len(ds["test"])))
print("pm25 (AQI) min/median/max: %.0f / %.0f / %.0f" % (pm.min(), np.median(pm), pm.max()))

row = train[0]
img = row["image"]
img = img if isinstance(img, Image.Image) else Image.open(io.BytesIO(img)).convert("RGB")
plt.imshow(img); plt.title("pm25 (AQI) = %.0f" % row["pm25"]); plt.axis("off"); plt.show()

## Done ✓

**Next:** open `notebooks/01_data_audit.ipynb`. Coming back later? Just re-run the
bootstrap cell (the dataset is already in Drive, so Step 2 is instant).